# NX STUDIO — moteur GPU gratuit sur Kaggle

Ce notebook fait tourner les moteurs IA de NX STUDIO sur le GPU gratuit de Kaggle (T4, 16 Go). Il se connecte **lui-même** à NX STUDIO : pas de tunnel ni d'adresse à configurer côté Kaggle.

**Une seule fois :**
1. *Settings → Accelerator* : **GPU T4 x2** (ou P100). *Settings → Internet* : **On** (Kaggle demande de vérifier ton numéro de téléphone).
2. *Add-ons → Secrets* : ajoute `NX_URL` (l'adresse publique de ton NX STUDIO) et `NX_GPU_AGENT_TOKEN` (le jeton défini dans `NX_GPU_AGENTS` côté NX STUDIO), et coche-les pour ce notebook.

**Ensuite, à chaque fois :** *Save Version → Save & Run All*. Le notebook tourne jusqu'à 12 h, même onglet fermé, et consomme ton quota de 30 h par semaine. Pour l'arrêter : *View Active Events → Stop*.


In [ ]:
# Choix des moteurs pour cette session (la mémoire de Kaggle ne permet pas de tout charger à la fois)
ENGINES = "ltx-video,real-esrgan"     # vidéo + upscale
# ENGINES = "flux,real-esrgan"        # images + upscale
# ENGINES = "wan-5b"                  # vidéo Wan 5B : plus fidèle, mais 20 min et plus par clip sur T4
REPO = "https://github.com/djnastx-a11y/NX-IMAGE-VIDEO"


In [ ]:
%%bash -s "$REPO"
set -e
rm -rf /kaggle/working/nx && git clone --depth 1 "$1" /kaggle/working/nx
pip install -q -r /kaggle/working/nx/gpu-worker/requirements-kaggle.txt
python -c "import torch; print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN — active le GPU dans Settings')"


In [ ]:
import os, subprocess, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
env = dict(os.environ,
    NX_URL=secrets.get_secret("NX_URL"),
    NX_GPU_AGENT_TOKEN=secrets.get_secret("NX_GPU_AGENT_TOKEN"),
    NX_ENGINES=ENGINES,
    HF_HOME="/tmp/hf",              # poids des modèles : hors de /kaggle/working (limité à 20 Go)
    NX_GPU_MODELS_DIR="/tmp/nx-models",
    NX_GPU_DATA_DIR="/tmp/nx-gpu",
    NX_GPU_OFFLOAD="model",
    PYTHONUNBUFFERED="1",
)
# Tourne jusqu'à l'arrêt de la session ; les logs s'affichent ici.
proc = subprocess.Popen([sys.executable, "-m", "nx_gpu.agent"], cwd="/kaggle/working/nx/gpu-worker", env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print("Agent arrêté, code", proc.returncode)
